In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import io
import tarfile
import gzip
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [2]:
df_path = Path('../data/feat_matrix/Manipulate-Image-Features.pkl')
archive_path = df_path.with_name(df_path.stem + '.tar.gz')

if not os.path.exists(archive_path):
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add(df_path, arcname=df_path.name)
else:
    with tarfile.open(archive_path, 'r:gz') as tar:
        pkl_members = [m for m in tar.getmembers()
                       if m.name.endswith('.pkl') and m.isfile()]
        
        if not pkl_members:
            raise ValueError('No .pkl file found in archive')
        
        member = pkl_members[0]
        print(f'Extracting: {member.name!r} size={member.size} bytes')

        if member.size == 0:
            raise ValueError('Pkl member has 0 bytes -- archive is corrupt, delete and re-run')
        
        f = tar.extractfile(member)
        if f is None:
            raise ValueError('Could not extract .pkl file')
            
        data = f.read()

    if data[:2] == b'\x1f\x8b':
        data = gzip.decompress(data)

    df = pickle.loads(data)

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'Expected DataFrame, got {type(df)}')

Extracting: 'Manipulate-Image-Features.pkl' size=7694127320 bytes


In [33]:
df_sample = df[:30000]
df_sample.head(5)

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
0,0,0,0.606445,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.064453,0.263672,1.727823,0.852311,0.168868,0.581885,0.691629,-0.247261,0.835812,0.853667
1,0,0,0.634766,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.071289,0.261719,1.480847,0.845447,0.198830,0.624428,2.182204,-0.517785,0.817380,0.832598
2,0,0,0.635742,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.061523,0.257812,1.843750,0.891489,0.178098,0.617363,3.087025,-0.062009,0.867658,0.894150
3,0,0,0.609375,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.055664,0.244141,2.644153,0.841295,0.151001,0.560503,2.269908,-0.053270,0.814864,0.895385
4,0,0,0.616211,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.072266,0.256836,2.305444,0.842433,0.156008,0.552632,2.916367,-0.637114,0.826557,0.880141


In [51]:
X = df_sample.drop(['image_id', 'label'], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=70)
X_pca = pca.fit_transform(X_scaled)

In [58]:
df_pca = pd.DataFrame(X_pca, columns=[f'feat_{x}' for x in np.arange(70)])
df_pca.insert(0, 'image_id', df_sample['image_id'])
df_pca.insert(1, 'label', df_sample['label'])
df_pca.head(5)

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_60,feat_61,feat_62,feat_63,feat_64,feat_65,feat_66,feat_67,feat_68,feat_69
0,0,0,-2.655490,-2.997988,-0.489680,-0.667807,0.665371,1.502735,-0.266371,-0.413583,...,-0.017256,-0.276485,0.121277,0.187861,-0.047182,-0.112320,-0.237056,-0.090967,0.001184,-0.096479
1,0,0,-1.937455,-2.892971,-0.190486,-0.997067,0.535812,1.498628,-0.111303,-0.553599,...,0.003125,-0.302012,0.120499,0.253225,-0.170883,0.008513,-0.116612,0.033869,0.066516,0.006695
2,0,0,-2.017606,-2.781667,0.310487,-1.020113,0.568101,1.325062,0.335853,-0.665508,...,0.028573,0.031242,0.074465,0.177069,-0.017338,0.010530,-0.090110,0.098527,-0.013160,-0.009512
3,0,0,-2.653697,-2.764286,0.379104,-0.520357,0.651408,1.683023,0.053080,-0.384454,...,-0.050379,0.016927,-0.009689,0.137105,-0.226620,0.194664,-0.030238,0.106553,-0.007862,0.007112
4,0,0,-2.743554,-2.916764,-0.688075,-0.433478,0.481790,1.550892,-0.086446,-0.326008,...,-0.042557,-0.076048,-0.028266,-0.019227,-0.155008,0.272091,0.175556,0.117681,0.094652,0.067855


In [79]:
feat_cols = df_pca.filter(like='feat_').columns

C = df_pca.groupby('image_id').size()
C = pd.DataFrame(C, columns=['Count'])
D = df_pca.groupby('image_id')['label'].sum() / C['Count']
D = pd.DataFrame(D, columns=['Density'])
F = df_pca.groupby('image_id')[feat_cols].mean()

In [92]:
df_cluster = pd.concat([C, D, F], axis=1)
df_cluster

,Count,Density,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_60,feat_61,feat_62,feat_63,feat_64,feat_65,feat_66,feat_67,feat_68,feat_69
image_id,,,,,,,,,,,,,,,,,,,,,
0,4599,0.000000,-1.605006,-2.337562,0.232256,-1.218415,0.164290,0.505381,0.074208,-0.527881,...,-0.044689,0.031198,0.032155,-0.010793,-0.003389,0.000534,-0.001971,0.015773,0.036874,0.035571
1,7735,0.013963,3.343326,0.622087,-1.835024,-0.195877,-0.430285,-0.926428,-0.077759,0.327445,...,0.035361,-0.015188,0.007304,0.006048,-0.021917,-0.016417,-0.018451,-0.010512,-0.025309,-0.008422
2,2961,0.000000,0.704054,0.883528,0.539061,0.599036,0.359560,0.442042,-0.105216,-0.016610,...,0.017486,0.003026,-0.030842,0.022326,0.005147,0.024401,-0.015258,0.003502,0.005942,-0.025623
3,2961,0.101317,-0.325563,0.817454,0.579764,0.832616,0.378833,0.374092,0.017744,0.020915,...,-0.025763,-0.012257,-0.021127,0.011368,0.008503,0.031473,-0.023517,0.003017,-0.006559,-0.009246
4,2961,0.026005,0.541851,0.895201,0.422467,0.661712,0.329171,0.438456,0.004090,0.002801,...,0.026494,0.009097,-0.007750,0.014157,0.000142,0.022836,-0.022495,-0.001810,0.009602,-0.020327
5,841,0.000000,-2.980926,0.226459,2.487382,-1.005105,0.867026,-0.269792,-0.105330,0.126916,...,-0.019944,0.018938,-0.059581,-0.042506,-0.036738,0.002926,0.041864,-0.043625,-0.044810,0.016807
6,841,0.231867,-2.888861,0.435652,2.169382,-0.891093,0.730561,-0.319063,0.064029,0.040848,...,-0.017494,0.025055,-0.061664,-0.048775,-0.030411,0.003141,0.052434,-0.041903,-0.028592,0.016567
7,2745,0.000000,-2.400594,-0.590935,0.645556,0.226003,-0.244568,0.216805,0.083018,-0.063230,...,-0.008218,-0.012923,0.005906,-0.013920,0.024853,-0.013997,0.039987,0.002614,0.007793,-0.003678
8,2745,0.154463,-2.312027,-0.092928,0.550395,0.428476,-0.267844,0.285204,0.124382,-0.072748,...,-0.005756,-0.005693,0.015239,-0.003496,0.031498,-0.026542,0.030052,0.019404,0.003771,0.012186


In [ ]:
def image_fingerprint(df):
    pass